In [0]:
import hashlib
import time
from datetime import datetime, timezone

import requests
from google.transit import gtfs_realtime_pb2
from pyspark.sql import types as T


# Unity Catalog configuration.
CATALOG = "bucharest_mobility"
BRONZE = "bronze"

TARGET_TABLE = f"{CATALOG}.{BRONZE}.gtfs_realtime_snapshots"

# The collector runs for a bounded period and polls each feed once per minute.
# These values can later be externalized when orchestration is introduced.
POLL_INTERVAL_SECONDS = 60
COLLECTION_MINUTES = 30

# Official TPBI GTFS-Realtime protobuf feeds.
FEEDS = {
    "trip_updates": "https://gtfs.tpbi.ro/api/gtfs-rt/tripUpdates",
    "vehicle_positions": "https://gtfs.tpbi.ro/api/gtfs-rt/vehiclePositions",
    "service_alerts": "https://gtfs.tpbi.ro/api/gtfs-rt/serviceAlerts",
}

In [0]:
# One Bronze row represents one raw GTFS-Realtime feed snapshot.
# The protobuf payload is preserved unchanged so Silver can parse it
# independently of the ingestion logic.
bronze_schema = T.StructType([
    T.StructField("feed_type", T.StringType(), False),
    T.StructField("feed_timestamp", T.LongType(), True),
    T.StructField("ingested_at", T.TimestampType(), False),
    T.StructField("payload_hash", T.StringType(), False),
    T.StructField("payload", T.BinaryType(), False),
    T.StructField("entity_count", T.IntegerType(), False),
    T.StructField("payload_size_bytes", T.LongType(), False),
])

In [0]:
# Create the Delta table once. Subsequent collection sessions append
# immutable snapshots to the existing history.
if not spark.catalog.tableExists(TARGET_TABLE):
    empty_df = spark.createDataFrame([], bronze_schema)

    (
        empty_df.write
        .format("delta")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"Created {TARGET_TABLE}")
else:
    print(f"{TARGET_TABLE} already exists")

In [0]:
def fetch_feed(feed_type, url):
    """
    Fetch one GTFS-Realtime protobuf response and attach ingestion metadata.

    The raw payload is retained for Bronze, while lightweight metadata is
    extracted to support observability and downstream processing.
    """
    response = requests.get(
        url,
        timeout=30,
    )

    response.raise_for_status()

    payload = response.content

    # Hash the raw protobuf bytes for content-based change detection.
    payload_hash = hashlib.sha256(payload).hexdigest()

    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(payload)

    feed_timestamp = (
        int(feed.header.timestamp)
        if feed.header.HasField("timestamp")
        else None
    )

    return {
        "feed_type": feed_type,
        "feed_timestamp": feed_timestamp,
        "ingested_at": datetime.now(timezone.utc),
        "payload_hash": payload_hash,
        "payload": payload,
        "entity_count": len(feed.entity),
        "payload_size_bytes": len(payload),
    }

In [0]:
def get_last_hashes():
    """
    Return the most recently ingested payload hash for each feed.

    Reading the state from Delta allows a new collection session to continue
    change detection across notebook runs instead of relying only on memory.
    """
    last_hashes = {}

    if not spark.catalog.tableExists(TARGET_TABLE):
        return last_hashes

    rows = spark.sql(f"""
        SELECT feed_type, payload_hash
        FROM (
            SELECT
                feed_type,
                payload_hash,
                ROW_NUMBER() OVER (
                    PARTITION BY feed_type
                    ORDER BY ingested_at DESC
                ) AS rn
            FROM {TARGET_TABLE}
        )
        WHERE rn = 1
    """).collect()

    for row in rows:
        last_hashes[row["feed_type"]] = row["payload_hash"]

    return last_hashes

In [0]:
def append_snapshot(snapshot):
    """Append one immutable raw feed snapshot to the Bronze Delta table."""
    df = spark.createDataFrame(
        [snapshot],
        schema=bronze_schema,
    )

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(TARGET_TABLE)
    )

In [0]:
# Recover the last observed payload for each feed before starting a new
# collection session. This prevents unchanged consecutive payloads from
# being appended across separate notebook runs.
last_hashes = get_last_hashes()

end_time = time.monotonic() + (COLLECTION_MINUTES * 60)

poll_number = 1

while time.monotonic() < end_time:
    print(f"\n--- Poll {poll_number} ---")

    for feed_type, url in FEEDS.items():
        try:
            snapshot = fetch_feed(feed_type, url)

            # Bronze uses raw payload equality for change detection.
            # Semantic interpretation and normalization belong in Silver.
            if last_hashes.get(feed_type) == snapshot["payload_hash"]:
                print(
                    f"SKIP {feed_type} | "
                    f"unchanged | "
                    f"{snapshot['entity_count']} entities"
                )
                continue

            append_snapshot(snapshot)

            last_hashes[feed_type] = snapshot["payload_hash"]

            print(
                f"LOADED {feed_type} | "
                f"{snapshot['entity_count']} entities | "
                f"{snapshot['payload_size_bytes']:,} bytes"
            )

        # A failure in one feed should not terminate collection
        # for the remaining GTFS-Realtime feeds.
        except Exception as exc:
            print(f"ERROR {feed_type}: {exc}")

    poll_number += 1

    remaining = end_time - time.monotonic()

    if remaining > 0:
        time.sleep(
            min(POLL_INTERVAL_SECONDS, remaining)
        )

print("\nCollection session finished.")

In [0]:
# Summarize the historical collection accumulated in Bronze.
display(
    spark.sql(f"""
        SELECT
            feed_type,
            COUNT(*) AS snapshots,
            MIN(ingested_at) AS first_ingestion,
            MAX(ingested_at) AS last_ingestion,
            SUM(entity_count) AS total_entities_observed,
            SUM(payload_size_bytes) AS total_bytes
        FROM {TARGET_TABLE}
        GROUP BY feed_type
        ORDER BY feed_type
    """)
)

In [0]:
# Inspect recent ingestion metadata without decoding the raw protobuf payload.
display(
    spark.sql(f"""
        SELECT
            feed_type,
            feed_timestamp,
            ingested_at,
            entity_count,
            payload_size_bytes,
            payload_hash
        FROM {TARGET_TABLE}
        ORDER BY ingested_at DESC
        LIMIT 20
    """)
)